# Real Estate Valuation — EDA + Linear Regression

**Dataset:** UCI Real Estate Valuation Data Set (Sindian District, New Taipei City, Taiwan)

**Goal:** Predict `Y house price of unit area` using property features such as age, distance to nearest MRT station, number of nearby convenience stores, and location (latitude/longitude).

**What we'll do:**
1. Upload and load the data
2. Explore it (shapes, types, missing values, distributions, correlations)
3. Clean it up (drop unnecessary columns)
4. Train a simple Linear Regression model with scikit-learn
5. Evaluate the model

> This notebook is written for beginners — every step has a short explanation.


## 1. Upload the Dataset

Run the cell below, then click **Choose Files** and select `real_estate_valuation.csv` from your computer.

*(This dataset is originally an Excel/UCI file, but here we use the CSV version so `files.upload()` works cleanly in Colab.)*


In [ ]:
from google.colab import files

uploaded = files.upload()  # click "Choose Files" and select real_estate_valuation.csv


In [ ]:
import pandas as pd
import numpy as np

# Get the uploaded filename automatically (in case it's named differently)
filename = list(uploaded.keys())[0]
df = pd.read_csv(filename)

print("File loaded:", filename)
df.head()


## 2. Exploratory Data Analysis (EDA)

Before building any model, we always want to understand the data first:
- How many rows/columns do we have?
- What are the column types?
- Are there missing values?
- What do the distributions look like?
- Are any features strongly correlated with the price (our target)?


### 2.1 Basic shape and info

In [ ]:
print("Shape (rows, columns):", df.shape)
print()
df.info()


**Column meanings (from the UCI dataset description):**
- `No` — just a row index/ID, not a real feature
- `X1 transaction date` — the date of sale, in decimal year format (e.g. 2013.250 = March 2013)
- `X2 house age` — age of the house in years
- `X3 distance to the nearest MRT station` — distance in meters to the nearest metro station
- `X4 number of convenience stores` — number of convenience stores within walking distance
- `X5 latitude` — geographic coordinate
- `X6 longitude` — geographic coordinate
- `Y house price of unit area` — **target**: price per unit area (our value to predict)


### 2.2 Summary statistics

In [ ]:
df.describe()


This gives us a quick sense of scale for each column — for example, `X3 distance to the nearest MRT station` ranges much wider than `X4 number of convenience stores`, which is why scaling can matter for some models (though plain Linear Regression handles this reasonably well).


### 2.3 Check for missing values

In [ ]:
df.isnull().sum()


No missing values here — this dataset is already clean, so we don't need to worry about imputation or dropping rows.


### 2.4 Check for duplicate rows

In [ ]:
print("Number of duplicate rows:", df.duplicated().sum())


### 2.5 Distribution of the target variable (house price)

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(7,4))
plt.hist(df['Y house price of unit area'], bins=30, color='steelblue', edgecolor='black')
plt.title("Distribution of House Price per Unit Area")
plt.xlabel("Price per unit area")
plt.ylabel("Count")
plt.show()


The target is roughly bell-shaped with a few high-price outliers on the right. Linear Regression doesn't strictly require a normal target, but it's good to know outliers exist — they can pull the regression line.


### 2.6 Distributions of the numeric features

In [ ]:
feature_cols = ['X2 house age', 'X3 distance to the nearest MRT station',
                'X4 number of convenience stores']

df[feature_cols].hist(figsize=(12, 6), bins=20, color='seagreen', edgecolor='black')
plt.tight_layout()
plt.show()


Notice `X3 distance to the nearest MRT station` is heavily right-skewed — most houses are close to a station, but a few are very far away. This is common with distance-based features.


### 2.7 Correlation with the target

In [ ]:
corr = df.drop(columns=['No']).corr()
corr['Y house price of unit area'].sort_values(ascending=False)


In [ ]:
import seaborn as sns

plt.figure(figsize=(8,6))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm")
plt.title("Correlation Heatmap")
plt.show()


**Key takeaways from correlation:**
- `X4 number of convenience stores` and `X3 distance to the nearest MRT station` are the strongest predictors of price — more stores nearby is associated with higher price, and greater distance to the MRT is associated with lower price.
- `X2 house age` has a mild negative correlation with price.
- `X1 transaction date`, `X5 latitude`, and `X6 longitude` have weaker linear correlation with price, but location can still matter — we'll keep latitude/longitude since real estate price is heavily location-driven, and simple correlation doesn't capture non-linear/spatial patterns well.


### 2.8 Scatter plots: strongest features vs. price

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12,4))

axes[0].scatter(df['X3 distance to the nearest MRT station'], df['Y house price of unit area'],
                alpha=0.5, color='darkorange')
axes[0].set_xlabel("Distance to nearest MRT station (m)")
axes[0].set_ylabel("Price per unit area")
axes[0].set_title("Distance to MRT vs Price")

axes[1].scatter(df['X4 number of convenience stores'], df['Y house price of unit area'],
                alpha=0.5, color='purple')
axes[1].set_xlabel("Number of convenience stores")
axes[1].set_ylabel("Price per unit area")
axes[1].set_title("Convenience Stores vs Price")

plt.tight_layout()
plt.show()


We can clearly see: as distance to the MRT station increases, price tends to drop. As the number of nearby convenience stores increases, price tends to rise. Both make intuitive sense for real estate.


## 3. Data Cleaning — Dropping Unnecessary Columns

- `No` is just a row identifier — it carries no real information about price, so we drop it.
- We'll keep all `X` features (`X1`–`X6`) since each one plausibly affects price, and correlation analysis didn't show anything so useless it should be excluded outright.


In [ ]:
df_clean = df.drop(columns=['No'])
df_clean.head()


## 4. Prepare Data for Modeling

We split the data into:
- **X** — the input features
- **y** — the target we want to predict (house price)

Then we split into a **training set** (to fit the model) and a **test set** (to evaluate it on unseen data).


In [ ]:
from sklearn.model_selection import train_test_split

X = df_clean.drop(columns=['Y house price of unit area'])
y = df_clean['Y house price of unit area']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print("Training set size:", X_train.shape)
print("Test set size:", X_test.shape)


## 5. Train a Linear Regression Model

We use scikit-learn's `LinearRegression`, which fits a straight-line relationship:

`price = w1*X1 + w2*X2 + ... + w6*X6 + b`

The model learns the weights (`w1`...`w6`) and bias (`b`) that best fit the training data.


In [ ]:
from sklearn.linear_model import LinearRegression

model = LinearRegression()
model.fit(X_train, y_train)

print("Model trained!")


### 5.1 Inspect the learned coefficients

In [ ]:
coef_df = pd.DataFrame({
    'Feature': X.columns,
    'Coefficient': model.coef_
}).sort_values(by='Coefficient', key=abs, ascending=False)

print("Intercept:", model.intercept_)
coef_df


**How to read this:** each coefficient tells us how much the predicted price changes for a 1-unit increase in that feature, holding all other features constant. For example, a positive coefficient on `X4 number of convenience stores` means more nearby stores is associated with a higher predicted price.


## 6. Evaluate the Model

We predict on the **test set** (data the model has never seen) and check how well it performs using:
- **R² score** — how much of the variance in price is explained by the model (closer to 1 is better)
- **RMSE** — root mean squared error, the typical size of our prediction error (in the same units as price)
- **MAE** — mean absolute error, the average absolute size of our prediction error


In [ ]:
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

y_pred = model.predict(X_test)

r2 = r2_score(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
mae = mean_absolute_error(y_test, y_pred)

print(f"R² score : {r2:.3f}")
print(f"RMSE     : {rmse:.3f}")
print(f"MAE      : {mae:.3f}")


A higher R² (closer to 1.0) means our features explain price well. RMSE and MAE tell us, on average, how far off our predictions are in the original price units — smaller is better.


### 6.1 Predicted vs Actual price

In [ ]:
plt.figure(figsize=(6,6))
plt.scatter(y_test, y_pred, alpha=0.6, color='teal')
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', label='Perfect prediction')
plt.xlabel("Actual price")
plt.ylabel("Predicted price")
plt.title("Predicted vs Actual House Price")
plt.legend()
plt.show()


If the model were perfect, every point would sit exactly on the red dashed line. Points scattered around the line (rather than randomly all over the plot) indicate the model is capturing a real, useful relationship — though not a perfect one.


### 6.2 Residual plot (errors)

In [ ]:
residuals = y_test - y_pred

plt.figure(figsize=(7,4))
plt.scatter(y_pred, residuals, alpha=0.6, color='crimson')
plt.axhline(y=0, color='black', linestyle='--')
plt.xlabel("Predicted price")
plt.ylabel("Residual (Actual - Predicted)")
plt.title("Residual Plot")
plt.show()


Residuals should ideally be scattered randomly around zero with no obvious pattern. If we see a funnel shape or curve, that's a sign Linear Regression's straight-line assumption may be too simple for this data — a common finding with real-world price data, but useful to know for future improvements (e.g. trying tree-based models).


## 7. Summary

- We loaded and explored the UCI Real Estate Valuation dataset (414 rows, 7 usable features).
- No missing values or duplicates were found — the data was already clean.
- `X4 number of convenience stores` and `X3 distance to the nearest MRT station` showed the strongest relationship with price.
- We dropped the `No` column (just an ID) and kept the rest as features.
- A simple Linear Regression model was trained and evaluated using R², RMSE, and MAE.

**Possible next steps (beyond beginner scope):**
- Try removing extreme outliers in price or distance
- Try polynomial features or a tree-based model (e.g. Random Forest) to capture non-linear relationships
- Use cross-validation instead of a single train/test split for a more robust performance estimate
